<a href="https://colab.research.google.com/github/Kanishka170-max/kanishka_2026B03110118/blob/main/kanishka_2026B03110118.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Project 9: Hospital Appointment Operations Analysis
Uses only basic Python (csv module + built-ins). No pandas / NumPy.
Run:  python hospital_analysis.py
Input : dataset.csv  (09_hospital_appointments.csv)
Output: cleaned_appointments.csv, analysis_report.txt
"""
import csv
from collections import defaultdict, Counter

INPUT = "dataset.csv"
CLEAN = "cleaned_appointments.csv"
REPORT = "analysis_report.txt"

lines = []


def out(text=""):
    print(text)
    lines.append(text)


# ---------- basic-python statistics helpers ----------
def mean(v):
    return sum(v) / len(v)


def percentile(sorted_v, p):
    """Linear-interpolation percentile (same as NumPy default)."""
    k = (len(sorted_v) - 1) * p / 100
    lo = int(k)
    hi = min(lo + 1, len(sorted_v) - 1)
    return sorted_v[lo] + (sorted_v[hi] - sorted_v[lo]) * (k - lo)


def median(v):
    return percentile(sorted(v), 50)


def std_dev(v):
    m = mean(v)
    return (sum((x - m) ** 2 for x in v) / len(v)) ** 0.5  # population SD (NumPy default)


def pct(a, b):
    return round(100 * a / b, 2) if b else 0.0


# ---------- 1. Load and inspect ----------
with open(INPUT, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

out("=" * 60)
out("1. LOAD AND INSPECT")
out("=" * 60)
out(f"Rows: {len(rows)}   Columns: {len(rows[0])}")
out("Columns: " + ", ".join(rows[0].keys()))
for r in rows[:3]:
    out("  " + str(dict(r)))

# ---------- 2. Missing waiting times ----------
out("\n" + "=" * 60)
out("2. MISSING WAITING TIMES")
out("=" * 60)
missing = [r for r in rows if r["Wait_Time_Min"].strip() == ""]
out(f"Missing Wait_Time_Min: {len(missing)}")
out("Affected IDs: " + ", ".join(r["Appointment_ID"] for r in missing))

# Standardise entries so every record matches the rest
for r in rows:
    for col in ("Department", "Appointment_Type", "SMS_Reminder", "Attendance_Status"):
        r[col] = r[col].strip()
    r["Department"] = r["Department"].title()
    r["SMS_Reminder"] = r["SMS_Reminder"].capitalize()
    # dates: M/D/YYYY -> YYYY-MM-DD so all dates share one fixed-width format
    m, d, y = r["Appointment_Date"].strip().split("/")
    r["Appointment_Date"] = f"{y}-{int(m):02d}-{int(d):02d}"
    r["Age"] = int(r["Age"])
    r["Consultation_Fee"] = int(r["Consultation_Fee"])

# Fill missing waits with the median wait of the same department
dept_known = defaultdict(list)
for r in rows:
    if r["Wait_Time_Min"].strip() != "":
        dept_known[r["Department"]].append(int(r["Wait_Time_Min"]))
dept_median = {d: median(v) for d, v in dept_known.items()}
for r in rows:
    if r["Wait_Time_Min"].strip() == "":
        r["Wait_Time_Min"] = round(dept_median[r["Department"]])
        r["Wait_Imputed"] = "Yes"
    else:
        r["Wait_Time_Min"] = int(r["Wait_Time_Min"])
        r["Wait_Imputed"] = "No"
out("Method: filled with the median wait of the same department (robust to outliers).")
out("Department medians used: " + ", ".join(f"{d}={dept_median[d]:.0f}" for d in sorted(dept_median)))

fields = ["Appointment_ID", "Age", "Department", "Appointment_Date", "Wait_Time_Min",
          "Appointment_Type", "SMS_Reminder", "Attendance_Status", "Consultation_Fee", "Wait_Imputed"]
with open(CLEAN, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(rows)
out(f"Cleaned data saved to {CLEAN}")

# ---------- 3. Duplicates ----------
out("\n" + "=" * 60)
out("3. DUPLICATE APPOINTMENT_ID CHECK")
out("=" * 60)
id_counts = Counter(r["Appointment_ID"] for r in rows)
dups = {k: v for k, v in id_counts.items() if v > 1}
out(f"Duplicate IDs found: {len(dups)}" + (f" -> {dups}" if dups else " (all IDs are unique)"))

# ---------- 4. Totals and attendance % ----------
out("\n" + "=" * 60)
out("4. TOTAL APPOINTMENTS AND ATTENDANCE STATUS %")
out("=" * 60)
total = len(rows)
status = Counter(r["Attendance_Status"] for r in rows)
out(f"Total appointments: {total}")
for s, c in status.most_common():
    out(f"  {s:<10} {c:>4}  {pct(c, total):>6}%")

# ---------- 5. Avg wait by department ----------
out("\n" + "=" * 60)
out("5. AVERAGE WAITING TIME BY DEPARTMENT (min)")
out("=" * 60)
by_dept = defaultdict(list)
for r in rows:
    by_dept[r["Department"]].append(r)
dept_wait = {d: mean([r["Wait_Time_Min"] for r in v]) for d, v in by_dept.items()}
for d, w in sorted(dept_wait.items(), key=lambda x: -x[1]):
    out(f"  {d:<12} {w:6.2f}   (n={len(by_dept[d])})")

# ---------- 6. No-show % by department ----------
out("\n" + "=" * 60)
out("6. NO-SHOW % BY DEPARTMENT (highest first)")
out("=" * 60)
dept_ns = {}
for d, v in by_dept.items():
    n = sum(1 for r in v if r["Attendance_Status"] == "No-show")
    dept_ns[d] = (n, len(v), pct(n, len(v)))
for d, (n, t, p) in sorted(dept_ns.items(), key=lambda x: -x[1][2]):
    out(f"  {d:<12} {n:>3} / {t:<3} = {p:>5}%")

# ---------- 7. SMS reminder vs attendance ----------
out("\n" + "=" * 60)
out("7. SMS REMINDER vs ATTENDANCE (crosstab)")
out("=" * 60)
statuses = ["Attended", "No-show", "Cancelled"]
out(f"  {'SMS':<6}" + "".join(f"{s:>11}" for s in statuses) + f"{'Total':>8}{'No-show %':>12}")
sms_ns = {}
for flag in ("Yes", "No"):
    sub = [r for r in rows if r["SMS_Reminder"] == flag]
    cnt = Counter(r["Attendance_Status"] for r in sub)
    sms_ns[flag] = pct(cnt["No-show"], len(sub))
    out(f"  {flag:<6}" + "".join(f"{cnt[s]:>11}" for s in statuses) + f"{len(sub):>8}{sms_ns[flag]:>11}%")
out(f"  Difference (No minus Yes): {round(sms_ns['No'] - sms_ns['Yes'], 2)} percentage points")

# ---------- 8. Avg fee by department ----------
out("\n" + "=" * 60)
out("8. AVERAGE CONSULTATION FEE BY DEPARTMENT")
out("=" * 60)
for d, v in sorted(by_dept.items(), key=lambda x: -mean([r['Consultation_Fee'] for r in x[1]])):
    out(f"  {d:<12} {mean([r['Consultation_Fee'] for r in v]):8.2f}")

# ---------- 9. Volume by date and department ----------
out("\n" + "=" * 60)
out("9. APPOINTMENT VOLUME BY DATE AND DEPARTMENT")
out("=" * 60)
depts = sorted(by_dept)
month_names = {"01": "Jan", "02": "Feb", "03": "Mar", "04": "Apr", "05": "May", "06": "Jun"}
by_month = defaultdict(Counter)
by_date = Counter()
by_date_dept = defaultdict(Counter)
for r in rows:
    mth = r["Appointment_Date"][5:7]
    by_month[mth][r["Department"]] += 1
    by_date[r["Appointment_Date"]] += 1
    by_date_dept[r["Appointment_Date"]][r["Department"]] += 1
out(f"  {'Month':<7}" + "".join(f"{d[:10]:>12}" for d in depts) + f"{'Total':>8}")
for mth in sorted(by_month):
    out(f"  {month_names.get(mth, mth):<7}" + "".join(f"{by_month[mth][d]:>12}" for d in depts)
        + f"{sum(by_month[mth].values()):>8}")
out(f"\n  Distinct dates: {len(by_date)}   Range: {min(by_date)} to {max(by_date)}")
out(f"  Average appointments per date: {mean(list(by_date.values())):.2f}")
out("  Top 5 busiest dates:")
for dt, c in by_date.most_common(5):
    brk = ", ".join(f"{d}={n}" for d, n in by_date_dept[dt].most_common())
    out(f"    {dt}  {c} appointments  ({brk})")

# ---------- 10. New vs follow-up ----------
out("\n" + "=" * 60)
out("10. WAITING TIME: NEW vs FOLLOW-UP")
out("=" * 60)
for t in ("New", "Follow-up"):
    w = [r["Wait_Time_Min"] for r in rows if r["Appointment_Type"] == t]
    out(f"  {t:<10} n={len(w):<4} mean={mean(w):6.2f}  median={median(w):5.1f}")

# ---------- 11. Median, quartiles, SD ----------
out("\n" + "=" * 60)
out("11. WAITING TIME STATISTICS (basic Python, NumPy-equivalent)")
out("=" * 60)
waits = sorted(r["Wait_Time_Min"] for r in rows)
q1, q2, q3 = percentile(waits, 25), percentile(waits, 50), percentile(waits, 75)
out(f"  Mean   : {mean(waits):.2f}")
out(f"  Median : {q2:.2f}")
out(f"  Q1     : {q1:.2f}")
out(f"  Q3     : {q3:.2f}")
out(f"  Std dev: {std_dev(waits):.2f}")
out(f"  Min/Max: {waits[0]} / {waits[-1]}")

# ---------- 12. IQR outliers ----------
out("\n" + "=" * 60)
out("12. UNUSUALLY LONG WAITS (IQR METHOD)")
out("=" * 60)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
out(f"  IQR = {iqr:.2f}   Lower fence = {lower:.2f}   Upper fence = {upper:.2f}")
high = [r for r in rows if r["Wait_Time_Min"] > upper]
low = [r for r in rows if r["Wait_Time_Min"] < lower]
out(f"  Unusually long waits (> {upper:.1f} min): {len(high)}")
for r in sorted(high, key=lambda x: -x["Wait_Time_Min"]):
    out(f"    {r['Appointment_ID']}  {r['Department']:<12} {r['Wait_Time_Min']} min  {r['Attendance_Status']}")
out(f"  Unusually short waits (< {lower:.1f} min): {len(low)}")

# ---------- 13. Waiting-time categories ----------
out("\n" + "=" * 60)
out("13. WAITING-TIME CATEGORIES")
out("=" * 60)


def band(w):
    if w <= 15:
        return "Short (0-15)"
    if w <= 30:
        return "Moderate (16-30)"
    if w <= 45:
        return "Long (31-45)"
    return "Very long (46+)"


band_order = ["Short (0-15)", "Moderate (16-30)", "Long (31-45)", "Very long (46+)"]
band_count = Counter(band(r["Wait_Time_Min"]) for r in rows)
for b in band_order:
    out(f"  {b:<18} {band_count[b]:>4}  {pct(band_count[b], total):>6}%")
out("  Share of Long + Very long waits by department:")
for d in depts:
    v = by_dept[d]
    n = sum(1 for r in v if band(r["Wait_Time_Min"]) in band_order[2:])
    out(f"    {d:<12} {n:>3} / {len(v):<3} = {pct(n, len(v)):>5}%")

# ---------- 14. Estimated fees for attended ----------
out("\n" + "=" * 60)
out("14. ESTIMATED CONSULTATION FEES (ATTENDED ONLY)")
out("=" * 60)
attended = [r for r in rows if r["Attendance_Status"] == "Attended"]
fee_total = sum(r["Consultation_Fee"] for r in attended)
out(f"  Attended appointments: {len(attended)}")
out(f"  Estimated total fees : {fee_total:,}")
out(f"  Average fee/attended : {fee_total / len(attended):.2f}")
out(f"  {'Department':<12}{'Attended':>10}{'Total fees':>14}")
for d in depts:
    a = [r for r in attended if r["Department"] == d]
    out(f"  {d:<12}{len(a):>10}{sum(r['Consultation_Fee'] for r in a):>14,}")
lost = sum(r["Consultation_Fee"] for r in rows if r["Attendance_Status"] in ("No-show", "Cancelled"))
out(f"  Fee value attached to no-show/cancelled slots: {lost:,}")

# ---------- 15. Recommendations ----------
out("\n" + "=" * 60)
out("15. OPERATIONAL RECOMMENDATIONS")
out("=" * 60)
worst_wait = max(dept_wait, key=dept_wait.get)
worst_ns = max(dept_ns, key=lambda d: dept_ns[d][2])
recs = [
    f"Target no-show reduction in {worst_ns} ({dept_ns[worst_ns][2]}% no-show, the highest): "
    f"use confirmation calls and a waitlist to refill missed slots.",
    f"SMS reminders: no-show was {sms_ns['Yes']}% with SMS vs {sms_ns['No']}% without. "
    f"Send reminders to every patient and test a second reminder closer to the visit.",
    f"Review scheduling in {worst_wait} (longest average wait, {dept_wait[worst_wait]:.1f} min): "
    f"stagger start times and check clinic session length.",
    f"Investigate the {len(high)} unusually long waits (above {upper:.0f} min) for common causes "
    f"such as late doctor start, overbooking or specific dates, and set an alert threshold.",
    f"Plan staffing around peak dates (up to {by_date.most_common(1)[0][1]} appointments on "
    f"{by_date.most_common(1)[0][0]}) and use overbooking only where no-show/cancellation rates are high.",
]
for i, t in enumerate(recs, 1):
    out(f"  {i}. {t}")
out("\nNote: operational analysis only; no clinical conclusions are drawn.")

with open(REPORT, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

1. LOAD AND INSPECT
Rows: 500   Columns: 9
Columns: Appointment_ID, Age, Department, Appointment_Date, Wait_Time_Min, Appointment_Type, SMS_Reminder, Attendance_Status, Consultation_Fee
  {'Appointment_ID': 'AP00000', 'Age': '18', 'Department': 'General', 'Appointment_Date': '5/4/2025', 'Wait_Time_Min': '37', 'Appointment_Type': 'Follow-up', 'SMS_Reminder': 'Yes', 'Attendance_Status': 'Attended', 'Consultation_Fee': '1200'}
  {'Appointment_ID': 'AP00001', 'Age': '32', 'Department': 'Pediatrics', 'Appointment_Date': '1/22/2025', 'Wait_Time_Min': '31', 'Appointment_Type': 'Follow-up', 'SMS_Reminder': 'Yes', 'Attendance_Status': 'Attended', 'Consultation_Fee': '1200'}
  {'Appointment_ID': 'AP00002', 'Age': '46', 'Department': 'General', 'Appointment_Date': '2/26/2025', 'Wait_Time_Min': '33', 'Appointment_Type': 'New', 'SMS_Reminder': 'Yes', 'Attendance_Status': 'Attended', 'Consultation_Fee': '700'}

2. MISSING WAITING TIMES
Missing Wait_Time_Min: 10
Affected IDs: AP00125, AP00158, AP0019